# 8 · Widgets — using every control + adding your own

Widgets describe how a node parameter renders in the UI. One class per `WidgetType` enum value means a generic frontend can render any node by reading the registry.

In this notebook:

1. Zero-ceremony defaults — just type hints
2. Explicit widgets for control and constraints
3. The full widget catalog — one example each
4. Inspecting the schema (what the frontend actually sees)
5. Writing your own widget class when the catalog isn't enough

In [ ]:
from typing import Annotated

from conductor import GraphNode, NodeRegistry, compile
from conductor.execution.engine import collect, execute
from conductor.registry.schema import serialize_registry
from conductor.widgets import (
    Checkbox, CodeEditor, DatePicker, Dropdown, DependentDropdown, EntityDropdown,
    FileUpload, IfElseBuilder, List, Multiselect, Number, Output, Range,
    SchemaBuilder, Switch, TemplateTextarea, Text, Textarea,
)
from conductor.types import Base64Str, Date

import json

## 1 · Type defaults — the easy path

If a parameter has only a type hint, the registry picks a default widget:

| You write | You get |
|---|---|
| `x: str` | `Text` |
| `x: int` | `Number(integer_only=True)` |
| `x: float` | `Number` |
| `x: bool` | `Checkbox` |
| `x: Date` | `DatePicker` |
| `x: list[str]` | `List(item_widget=Text)` |
| `x: list[int]` | `List(item_widget=Number(integer_only=True))` |
| `x: dict` | `SchemaBuilder` |

Annotate explicitly when you want constraints, a different widget, or a human-friendly label.

In [ ]:
reg_defaults = NodeRegistry()


@reg_defaults.node("plan-trip", version=1, name="Plan Trip", description="Plan a trip")
def plan_trip(
    days: int,
    include_weekends: bool,
    destinations: list[str],
    preferences: dict,
) -> Annotated[str, Output(label="Plan")]:
    return f"{days} days, weekends={include_weekends}, to {destinations}, prefs={preferences}"


# Inspect the registered shape — no widget annotations written, yet every
# input has a widget and the right constraints.
node_def = reg_defaults.get("plan-trip@1")
for inp in node_def.inputs:
    cfg_keys = list(inp.widget_config.keys()) if inp.widget_config else []
    print(f"  {inp.name:<15} {inp.type_str:<12} widget={inp.widget.value:<14} config={cfg_keys}")

## 2 · Explicit widgets — when you want control

Use `Annotated[T, Widget(...)]` when you care about the control type, validation constraints, or UX labels.

In [ ]:
reg_explicit = NodeRegistry()


@reg_explicit.node("send-email", version=1, name="Send Email", description="Send an email")
def send_email(
    to: Annotated[str, Text(label="Recipient", pattern=r"^[^@]+@[^@]+$")],
    subject: Annotated[str, Text(label="Subject", max_length=200)],
    body: Annotated[str, Textarea(label="Body", rows=8, min_length=10)],
    priority: Annotated[str, Dropdown(label="Priority", choices=["low", "normal", "high"])] = "normal",
    retries: Annotated[int, Range(label="Retry attempts", min_val=0, max_val=5, step=1)] = 1,
) -> Annotated[str, Output(label="Status")]:
    return f"sent: {to=}, {subject=}, {priority=}, {retries=}"


for inp in reg_explicit.get("send-email@1").inputs:
    print(f"  {inp.name:<10} widget={inp.widget.value:<10} label={inp.label!r}")

## 3 · The full catalog — one example per widget

All 19 widget classes, each with a representative configuration. Every one of these renders from the registry JSON without any backend-specific code on the frontend.

In [ ]:
reg_all = NodeRegistry()


@reg_all.node("widget-showcase", version=1, name="Showcase", description="Every widget")
def showcase(
    # Text & code
    title:        Annotated[str,  Text(label="Title", max_length=100)],
    notes:        Annotated[str,  Textarea(label="Notes", rows=6)],
    template:     Annotated[str,  TemplateTextarea(label="Template", variables=["name", "date"])],
    code_snippet: Annotated[str,  CodeEditor(label="Code", language="python")],

    # Choice
    style:   Annotated[str,       Dropdown(label="Style", choices=["A", "B", "C"])],
    tier:    Annotated[str,       DependentDropdown(label="Tier", depends_on="style",
                                                    choices_map={"A": ["a1", "a2"], "B": ["b1"]})],
    tags:    Annotated[list[str], Multiselect(label="Tags", choices=["red", "blue", "green"])],
    user:    Annotated[str,       EntityDropdown(label="User", entity_kind="user", multiple=False)],

    # Numeric
    count:   Annotated[int,   Number(label="Count", integer_only=True, min_val=0)],
    volume:  Annotated[float, Range(label="Volume", min_val=0, max_val=1, step=0.01)],

    # Boolean
    enabled: Annotated[bool, Checkbox(label="Enabled")],
    darkmode: Annotated[bool, Switch(label="Dark mode")],

    # Date & file
    starts:  Annotated[str, DatePicker(label="Starts", min_date="2024-01-01")],
    upload:  Annotated[str, FileUpload(label="Upload", accept=".pdf,.csv", max_size_mb=10)],

    # Structured
    items:   Annotated[list[str], List(label="Items", item_widget=Text(label="item"),
                                        min_items=1, max_items=10)],
    config:  Annotated[dict,      SchemaBuilder(label="Config",
                                                schema={"type": "object"}, allow_additional=True)],
    branch:  Annotated[dict,      IfElseBuilder(label="Routing", variables=["count", "enabled"])],
) -> Annotated[str, Output(label="Summary")]:
    return "all good"


print(f"Registered {len(reg_all.get('widget-showcase@1').inputs)} inputs across every widget type.")

## 4 · Inspecting the schema — what the frontend sees

`widget.to_schema()` gives you the JSON for one widget. `serialize_registry(reg)` gives you the full per-node schema for the entire registry.

In [ ]:
w = Range(label="Volume", min_val=0, max_val=100, step=5, description="0–100")
print(json.dumps(w.to_schema(), indent=2))

In [ ]:
# Full node schema — this is exactly what conductor_providers.react.palette_from_registry returns.
schema = serialize_registry(reg_defaults)
plan_trip_schema = next(s for s in schema if s["id"] == "plan-trip@1")

# Show a trimmed view so the output stays readable
print(f"{plan_trip_schema['id']} — {plan_trip_schema['name']}")
for inp in plan_trip_schema["inputs"]:
    print(f"  {inp['name']:<15} widget={inp['widget']:<14} label={inp['label']!r}")

## 5 · Writing your own widget

If the catalog isn't enough, a new widget is four steps — all local, no framework to fight. The pattern below is what every built-in widget follows.

In [ ]:
# Step 1 — imagine we need a hex color picker. In production this value would
#          go into types.py; for a notebook demo we can just use a string
#          constant that matches whatever the frontend dispatches on.

COLOR_PICKER_WIDGET_TYPE = "color-picker"


# Step 2 — define the Widget class.

from dataclasses import dataclass
from typing import Any
from conductor.widgets import Widget


@dataclass
class ColorPicker(Widget):
    """Hex color input. Value is a string like '#3366ff'."""

    disable_handle: bool = True
    default: str | None = None

    @property
    def widget_type(self) -> Any:            # str is fine here; real widget types use the enum
        return COLOR_PICKER_WIDGET_TYPE

    def to_schema(self) -> dict[str, Any]:
        schema = super().to_schema()
        if self.default is not None:
            schema["default"] = self.default
        return schema


# Step 3 — use it in a node exactly like any built-in widget.

reg_custom = NodeRegistry()


@reg_custom.node("themed-banner", version=1, name="Themed Banner", description="A banner")
def themed_banner(
    text: Annotated[str, Text(label="Text")],
    color: Annotated[str, ColorPicker(label="Background", default="#3366ff")],
) -> Annotated[str, Output(label="Rendered")]:
    return f"<span style='background:{color}'>{text}</span>"


inp = reg_custom.get("themed-banner@1").inputs[1]
print(f"name={inp.name}  widget={inp.widget}  label={inp.label}")
print(json.dumps(inp.widget_config, indent=2))

### Production checklist

In a real addition (as opposed to this notebook demo):

1. Add the string to `conductor.types.WidgetType` so the enum is authoritative.
2. Put the class in `conductor.widgets` so users import it via `from conductor.widgets import ...`.
3. If it's the natural default for some Python type, wire it into `_default_widget` in `conductor.registry`.
4. Write a small test class in `tests/test_core/test_widget_defaults.py` — every built-in widget has one; copy a nearby one as a template.
5. Ship a matching frontend component that dispatches on your widget type string.

Reference: [`docs/widgets.md`](../docs/widgets.md) for the full recipe and conventions.

## 6 · Running a flow with only defaults

Zero widget annotations — the registry picks sensible ones, and the flow just runs.

In [ ]:
reg_run = NodeRegistry()


@reg_run.node("fmt", version=1, name="Format", description="Glue")
def fmt(n: int, label: str) -> Annotated[str, Output(label="Out")]:
    return f"{label}={n}"


compiled = compile(
    nodes=[GraphNode("n", "fmt@1", {"n": 42, "label": "answer"})],
    edges=[],
    registry=reg_run,
)
results = await collect(execute(compiled))
print(results["n"]["result"])